In [46]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# 1. Process & Explore data

In [47]:
# We need to extract the times from the data files 
# and create a matrix using (O,D) pairs with times
tt1 = pd.read_csv('travel_times_15_17.csv')
tt2 = pd.read_csv('travel_times_17_19.csv')
tt3 = pd.read_csv('travel_times_19_21.csv')
tt1.head()

,departure_time,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
0,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589152,"Fribourg, Mon-Repos",46.806711,7.172136,270,35,0.58,42,0.70,OK
1,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589138,"Fribourg, Cité-Jardins",46.809385,7.170446,659,86,1.43,117,1.95,OK
2,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.30,174,2.90,OK
3,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8587255,"Fribourg, Tilleul/Cathédrale",46.806090,7.161261,3788,445,7.42,506,8.43,OK
4,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589161,"Fribourg, St-Pierre",46.803911,7.155266,4335,561,9.35,622,10.37,OK


* Set with origins and set with destinations from data above

In [48]:
""" # let's use indexes 0 to n
origin_1 = tt1['origin_name']
destination_1 = tt1['dest_name']

origin_idx = {idx: o for idx, o i"""  """n enumerate(origin_1)}
dest_idx = {idx: d for idx, d in enumerate(destination_1)}
print(origin_idx) """

" # let's use indexes 0 to n\norigin_1 = tt1['origin_name']\ndestination_1 = tt1['dest_name']\n\norigin_idx = {idx: o for idx, o in enumerate(origin_1)}\ndest_idx = {idx: d for idx, d in enumerate(destination_1)}\nprint(origin_idx) "

* Matrix with (O,D) pairs: time I think (distance maybe?)
* MILP only optimize every x minutes so only take x minutes

In [49]:
""" time_window = 30 #min optimizing MILP window

# 1. Define your total time window 
start_time = tt1['departure_time'].min().floor('15min')
end_time = tt1['departure_time'].max().ceil('15min')

time_slots = pd.date_range(start=start_time, end=end_time,freq=f'{time_window}min')

# Loop for time_slots
for current_time in time_slots:
    # Cutoff of window
    next_time = current_time + pd.Timedelta(minutes=f"{time_window}")
    
    # extract trips that happen in the 15 minutes blocks
    mask = (tt1['deparure_time'] >= current_time) & (tt1['departure_time'] < next_time)
    window_df = 

od_matrices_by_time = {}
print(tt1['departuÆre_time'])



od_matrix = tt1.pivot(index='origin_name', 
                            columns='dest_name', 
                            values='duration_seconds')

print(od_matrix) """

' time_window = 30 #min optimizing MILP window\n\n# 1. Define your total time window \nstart_time = tt1[\'departure_time\'].min().floor(\'15min\')\nend_time = tt1[\'departure_time\'].max().ceil(\'15min\')\n\ntime_slots = pd.date_range(start=start_time, end=end_time,freq=f\'{time_window}min\')\n\n# Loop for time_slots\nfor current_time in time_slots:\n    # Cutoff of window\n    next_time = current_time + pd.Timedelta(minutes=f"{time_window}")\n\n    # extract trips that happen in the 15 minutes blocks\n    mask = (tt1[\'deparure_time\'] >= current_time) & (tt1[\'departure_time\'] < next_time)\n    window_df = \n\nod_matrices_by_time = {}\nprint(tt1[\'departuÆre_time\'])\n\n\n\nod_matrix = tt1.pivot(index=\'origin_name\', \n                            columns=\'dest_name\', \n                            values=\'duration_seconds\')\n\nprint(od_matrix) '

# 2. Define parameters

In [ ]:
# Define parameters
time_window = 30  # minutes for optimization window
num_buses = 5     # Number of available buses
bus_capacity =  [4, 6, 8]  # Passengers per bus
buses = [0, 1, 2] # index for buses

# Cost parameters
b1 = 1.0                    # Weight: TO BE DEFINED
b2 = 1                      # Weight: TO BE DEFINED
b3 = 0.5                    # Weight: TO BE DEFINED
cost_per_bus = [20,50,70]   # Cost per bus per minute of operation
cost_reject = 100           # Penalty for rejecting a trip


# 3. Model definition

* Chnge of plan: now create create window and loop throught wikndow

In [51]:
# Create a copy to preserve original data
tt1_copy = tt1.copy()

# Convert departure_time to datetime on the copy
tt1_copy['departure_time'] = pd.to_datetime(tt1_copy['departure_time'])

model = gp.Model("MILP")
trips = tt1_copy.index.tolist()

# Define your total time window 
start_time = tt1_copy['departure_time'].min().floor('15min')
end_time = tt1_copy['departure_time'].max().ceil('15min')
time_slots = pd.date_range(start=start_time, end=end_time, freq=f'{time_window}min')


# Loop for time_slots and optimize for each time slot
for current_time in time_slots[:-1]: # Exclude last endpoint
    # Cutoff of window
    next_time = current_time + pd.Timedelta(minutes=time_window)
    
    # extract trips that happen in the 15 minutes blocks
    mask = (tt1_copy['departure_time'] >= current_time) & (tt1_copy['departure_time'] < next_time)
    window_df = tt1_copy[mask].copy() # Apply mask to data
    window_trips = window_df.index.tolist()
    
    # Decision Variables
    # x decision variable if arc (i,j) is travelled so 0 or 1
    x = model.addVars(window_trips, buses, vtype=GRB.BINARY, name = "x")

    # Objective: SET OF BUSES k?
    # First we have the total operating costs
    total_costs = gp.quicksum(b1*cost_per_bus[k]*tt1_copy.loc[trip,'duration_seconds']*x[trip,k] 
                            for trip in window_trips
                            for k in buses)

    # Then there is a penalty for unserved customer
    customer_penalty = gp.quicksum(b2*cost_reject*(1-x[trip,k])
                                for trip in window_trips
                                for k in buses)


    # The future cost due to uncertainty
    E = 0 # TO BE DETERMINED
    cost_future =  b3*E

    # Minimize total
    model.setObjective(total_costs + customer_penalty + cost_future, GRB.MINIMIZE)